# Zero-shot zaman serisi anomali modeli — eğitim

Bu defter `cagrigungor/tisan-havuz` (özel HF veri seti) üzerindeki gerçek arka planı ve `egitim_verisi_uretici.py`
sentetik verisini birleştirerek iki eksenli dikkat modelini eğitir, kalibre eder, benchmark'ta ölçer ve
HF'ye `trust_remote_code` ile yükler.

**Veri rolleri**

| Rol | Kaynak | Kullanım |
|---|---|---|
| Eğitim | LOTSA, HAI train, CNC, SKAB teaser, wind `complex` | gerçek arka plan + sentetik anomali |
| Doğrulama | HAI test, wind `labeled`, Pump | model seçimi, kalibrasyon (temperature) |
| Benchmark | NAB, SMAP/MSL, SMD, SKAB | **sadece** final rapor, eğitime girmez |

İlk eğitimden çıkan dersler (v1, 20k adım): sentetik hücre AUC-PR 0.52 / AUC-ROC 0.93, ama gerçek doğrulamada AUC-ROC ~0.6 ve
6000. adımdan sonra düşüş → sentetik stile aşırı uyum + `max` toplulaştırmanın 60 sütunlu HAI'de yanlış pozitif üretmesi.
Bu sürümde: sentetik payı 0.4 → 0.2, enjeksiyon 0.75 → 0.5, doğrulama 60k → 250k satır, satır toplulaştırma seçimi (top-k / noisy-OR),
checkpoint seçimi (AUC-PR + AUC-ROC)/2, `INIT_FROM` ile önceki ağırlıklardan devam.

Colab'da: Çalışma zamanı → GPU (T4 yeter, A100 ideal). HF token'ı **Secrets** panelinde `HF_TOKEN` olarak tanımla; deftere yapıştırma.

In [ ]:
import os, sys, subprocess
if not os.path.exists("egitim_verisi_uretici.py"):
    subprocess.run(["git", "clone", "-q", "https://github.com/hasancagrigungor/tisan.git"], check=True)
    os.chdir("tisan")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.45", "huggingface_hub", "pyarrow", "pandas",
                "scikit-learn", "scipy", "tabulate", "matplotlib"], check=True)

from huggingface_hub import login, HfApi
try:
    from google.colab import userdata          # Colab Secrets
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except Exception:
    pass
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("HF:", HfApi().whoami()["name"])

## Ayarlar

In [ ]:
SMOKE = os.environ.get("SMOKE") == "1"          # yerel duman testi: küçük model, birkaç adım

REPO_DATA  = "cagrigungor/tisan-havuz"
REPO_MODEL = "cagrigungor/anomali-small"
PUSH       = os.environ.get("PUSH", "0" if SMOKE else "1") == "1"

CFG = dict(d_model=256, n_layers=6, n_heads=8, dropout=0.1, patch=16, max_t=2048, max_ch=100,
           extra_channels=["diff"])                 # ablation: [] ile kapat
STEPS        = 20_000
BATCH        = 16
LR           = 3e-4
WARMUP       = 500
EVAL_EVERY   = 1_000
P_SYNTHETIC  = 0.2        # örneklerin bu kadarı tamamen sentetik (0.4 → 0.2: sentetik stile aşırı uyumu azalt)
P_INJECT     = 0.5        # gerçek pencerelerin bu kadarına sentetik anomali enjekte edilir; kalanı temiz negatif
INIT_FROM    = os.environ.get("INIT_FROM", "")   # ör. "ckpt/best" veya HF repo: önceki eğitimden devam
CURRICULUM   = 0.6        # difficulty 0→1 bu oranda adımda tamamlanır
HOLDOUT_SECTORS = []      # leave-one-domain-out için ör. ["energy"]
MAX_CELLS_PER_SOURCE = 80_000_000   # RAM sınırı: kaynak başına satır×sütun
EVAL_MAX_ROWS = 250_000   # doğrulamada seri başına ilk N satır (HAI test ≈ 3 gün; 60k çok azdı)
SEED = 0

if SMOKE:
    CFG.update(d_model=64, n_layers=2, n_heads=4)
    STEPS, BATCH, EVAL_EVERY, WARMUP = 30, 4, 15, 5
    MAX_CELLS_PER_SOURCE, EVAL_MAX_ROWS = 3_000_000, 6_000

import torch, numpy as np, random
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEVICE == "cuda"
print("cihaz:", DEVICE, "| smoke:", SMOKE)

## Veri havuzu

In [ ]:
from huggingface_hub import snapshot_download
import pandas as pd
HAVUZ = "data/havuz"
if not os.path.exists(f"{HAVUZ}/katalog.csv"):
    snapshot_download(REPO_DATA, repo_type="dataset", local_dir=HAVUZ)
kat = pd.read_csv(f"{HAVUZ}/katalog.csv")
kat["benchmark"] = kat["benchmark"].fillna("")
kat["n_cells"] = kat.n_rows * kat.n_channels

is_bench = kat.benchmark != ""
is_val = ((kat.source == "hai") & kat.series_id.str.contains("/test")) | (kat.series_id == "wind_gearbox/labeled") | (kat.source == "pump")
is_hold = kat.sector.isin(HOLDOUT_SECTORS)
kat["rol"] = np.select([is_bench, is_val, is_hold], ["benchmark", "val", "holdout"], "train")
print(kat.groupby("rol").agg(seri=("series_id", "count"), satir_M=("n_rows", lambda x: round(x.sum() / 1e6, 1))))
print("eğitim sektörleri:", kat[kat.rol == "train"].sector.value_counts().to_dict())

In [ ]:
import pyarrow.parquet as pq

def load_series(rows, max_cells=None, rng=None):
    """Katalog satırlarını belleğe alır: {series_id: (t, X float32, labels int8)}. max_cells kaynak başına sınır."""
    rng = rng or np.random.default_rng(SEED)
    chosen = []
    for src, g in rows.groupby("source"):
        g = g.sample(frac=1, random_state=SEED)
        if max_cells:
            keep = g.n_cells.cumsum() <= max_cells
            keep.iloc[0] = True
            g = g[keep]
        chosen.append(g)
    chosen = pd.concat(chosen)
    out = {}
    for file, g in chosen.groupby("file"):
        ids = set(g.series_id)
        tbl = pq.read_table(f"{HAVUZ}/gercek/{file}", filters=[("series_id", "in", list(ids))]).to_pandas()
        tbl["series_id"] = tbl["series_id"].astype(str)
        meta = g.set_index("series_id")
        for sid, d in tbl.groupby("series_id", sort=False):
            T, k = int(meta.loc[sid, "n_rows"]), int(meta.loc[sid, "n_channels"])
            out[sid] = (d["timestamp"].to_numpy()[::k].astype(np.float64),
                        d["value"].to_numpy(np.float32).reshape(T, k),
                        d["label"].to_numpy(np.int8).reshape(T, k))
    return out

train_series = load_series(kat[kat.rol == "train"], MAX_CELLS_PER_SOURCE)
val_series   = load_series(kat[kat.rol == "val"])
tot = sum(v[1].size for v in train_series.values())
print(f"eğitim: {len(train_series)} seri, {tot/1e6:.0f}M hücre | doğrulama: {len(val_series)} seri")

## Eğitim örnekleri

Her örnek ya tamamen sentetik (`make_sample`) ya da gerçek bir arka plan penceresi: rastgele uzunluk (20–2048, yarısı tam),
rastgele çözünürlük düşürme, rastgele sütun alt kümesi, ardından üreticideki `generic_anomaly` ile enjekte edilen anomaliler.
Ön işleme `hf_model.modeling_anomali.prepare_window` — inference'taki `detect()` ile aynı kod.

In [ ]:
import math, warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
from egitim_verisi_uretici import make_sample, Ctx, generic_anomaly, TYPE_ID, MIN_T
import egitim_verisi_uretici as gen
from hf_model.modeling_anomali import prepare_window

MAX_T, MAX_CH, PATCH = CFG["max_t"], CFG["max_ch"], CFG["patch"]
gen.MAX_T = MAX_T                      # üretici ile aynı pencere

kat_train = kat[kat.rol == "train"].set_index("series_id").loc[list(train_series)]
_sectors = sorted(kat_train.sector.unique())
_by_sector = {s: kat_train[kat_train.sector == s] for s in _sectors}
_sector_w = {s: np.sqrt(g.n_rows.to_numpy()) for s, g in _by_sector.items()}

def _pick_series(rng):
    s = _sectors[rng.integers(len(_sectors))]              # sektörler dengeli
    g, w = _by_sector[s], _sector_w[s]
    return g.index[rng.choice(len(g), p=w / w.sum())]

def _downsample(t, X, lab, f):
    n = len(t) // f
    Xb = X[:n * f].reshape(n, f, -1)
    return t[:n * f:f], np.nanmean(Xb, axis=1), lab[:n * f].reshape(n, f, -1).max(1)

def real_item(rng, difficulty):
    for _ in range(20):
        t, X, lab = train_series[_pick_series(rng)]
        f = int(rng.choice([1, 1, 1, 2, 4, 8]))
        if len(t) // f < MIN_T * 2:
            f = 1
        if f > 1:
            t, X, lab = _downsample(t, X, lab, f)
        n, kc = X.shape
        L = MAX_T if rng.random() < 0.5 else int(rng.integers(MIN_T, MAX_T + 1))
        L = min(L, n)
        a = int(rng.integers(0, n - L + 1))
        k = min(kc, max(1, int(round(math.exp(rng.uniform(0, math.log(min(MAX_CH, kc)) if kc > 1 else 0))))))
        cols = rng.choice(kc, k, replace=False)
        Xw, lw, tw = X[a:a + L][:, cols].astype(np.float64), lab[a:a + L][:, cols].copy(), t[a:a + L].copy()
        ok = ~np.isnan(Xw).all(0) & (np.nanstd(Xw, axis=0) > 0)
        if not ok.any():
            continue
        Xw, lw = Xw[:, ok], lw[:, ok]
        Xw = gen_fill(Xw)
        k = Xw.shape[1]
        real_lab = (lw == 1).astype(np.int8)
        types = np.where(real_lab == 1, -1, 0).astype(np.int8)          # gerçek etiketin türü bilinmiyor
        if rng.random() < P_INJECT:
            keep = None
            if rng.random() < gen.MISSING_DATA_RATIO and L > 40:
                gap = int(rng.integers(5, min(100, L // 4)))
                gi = int(rng.integers(L // 10, L - gap - 1))
                keep = np.r_[0:gi, gi + gap:L]
                Xw, tw, real_lab, types = Xw[keep], tw[keep], real_lab[keep], types[keep]
            ctx = Ctx(rng, Xw, tw, ["unknown"] * k, np.zeros(k, dtype=int), difficulty, {}, None)
            if keep is not None:
                ctx.mark(gi, gi + 1, list(range(k)), "missing_data")
            for _ in range(int(rng.integers(1, 4))):
                generic_anomaly(ctx)
            Xw = ctx.X
            inj = ctx.labels == 1
            real_lab = np.maximum(real_lab, ctx.labels)
            types = np.where(inj, ctx.types, types)
        return pack(tw, Xw, real_lab, types)
    return synthetic_item(rng, difficulty)

def gen_fill(X):
    from hf_model.modeling_anomali import fill_nan
    return fill_nan(X)

def pack(t, X, labels, types):
    T, k = X.shape
    values, dtf, tm, cm = prepare_window(t, X, MAX_T, MAX_CH)
    lab = np.zeros((MAX_T, MAX_CH), dtype=np.int8); lab[:T, :k] = labels
    typ = np.zeros((MAX_T, MAX_CH), dtype=np.int8); typ[:T, :k] = types
    return dict(values=values, delta_t=dtf, time_mask=tm, channel_mask=cm, labels=lab, types=typ)

def synthetic_item(rng, difficulty):
    for _ in range(10):
        s = make_sample(rng, difficulty=difficulty)
        if np.isfinite(s["raw"]).all():
            break
    # üretici kendi normalize eder; NaN yok. Aynı ön işleme için ham matristen yeniden paketle.
    T, k = s["meta"]["T"], s["meta"]["k"]
    return pack(s["raw"][:, 0], s["raw"][:, 1:], s["labels"][:T, :k], s["types"][:T, :k])

class TrainStream(torch.utils.data.IterableDataset):
    def __init__(self, difficulty_fn):
        self.difficulty_fn = difficulty_fn
        self.step = 0
    def __iter__(self):
        wi = torch.utils.data.get_worker_info()
        rng = np.random.default_rng([SEED, wi.id if wi else 0, int(torch.initial_seed()) % 2**31])
        while True:
            d = self.difficulty_fn()
            yield synthetic_item(rng, d) if rng.random() < P_SYNTHETIC else real_item(rng, d)

def collate(items):
    b = {k: torch.from_numpy(np.stack([it[k] for it in items])) for k in items[0]}
    # dolgu sağda: geçerli en uzun T (patch katı) ve en çok sütuna kırp → hesap tasarrufu
    T_eff = int(math.ceil(int(b["time_mask"].sum(1).max()) / PATCH) * PATCH)
    C_eff = int(b["channel_mask"].sum(1).max())
    for k in ("values", "labels", "types"):
        b[k] = b[k][:, :T_eff, :C_eff].contiguous()
    b["delta_t"], b["time_mask"] = b["delta_t"][:, :T_eff], b["time_mask"][:, :T_eff]
    b["channel_mask"] = b["channel_mask"][:, :C_eff]
    return b

_rng = np.random.default_rng(1)
for _ in range(3):
    it = real_item(_rng, 0.5)
    print("gerçek:", int(it["time_mask"].sum()), "satır", int(it["channel_mask"].sum()), "sütun, anomali hücre", int(it["labels"].sum()))
it = synthetic_item(_rng, 0.5); print("sentetik:", int(it["time_mask"].sum()), "satır", int(it["channel_mask"].sum()), "sütun")

## Model

In [ ]:
from hf_model.configuration_anomali import AnomaliConfig
from hf_model.modeling_anomali import AnomaliModel

config = AnomaliConfig(**CFG)
if INIT_FROM:
    model = AnomaliModel.from_pretrained(INIT_FROM).to(DEVICE)
    print("ağırlıklar yüklendi:", INIT_FROM)
else:
    model = AnomaliModel(config).to(DEVICE)
print(f"parametre: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## Değerlendirme

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

from hf_model.modeling_anomali import aggregate_rows

_cell_cache = {}          # sid -> (prob, y): toplulaştırma karşılaştırması için

def eval_real(series, max_rows=None, batch_size=8, agg="topk", topk=3, cache=False):
    """Etiketli gerçek serilerde satır bazında AUC-PR / AUC-ROC / en iyi F1 (point-adjust YOK)."""
    model.eval()
    rows = []
    for sid, (t, X, lab) in series.items():
        if max_rows:
            t, X, lab = t[:max_rows], X[:max_rows], lab[:max_rows]
        y = (lab == 1).any(1).astype(int)
        if y.sum() == 0 or y.sum() == len(y):
            continue
        if cache and sid in _cell_cache:
            prob = _cell_cache[sid][0]
        else:
            prob, _ = model.score_matrix(t, X.astype(np.float64), batch_size=batch_size)
            if cache:
                _cell_cache[sid] = (prob, y)
        s = np.nan_to_num(aggregate_rows(prob, agg, topk), nan=0.0)
        p, r, _ = precision_recall_curve(y, s)
        rows.append(dict(series_id=sid, source=sid.split("/")[0], n=len(y), anom=float(y.mean()),
                         auc_pr=average_precision_score(y, s), auc_roc=roc_auc_score(y, s),
                         best_f1=float(np.max(2 * p * r / np.maximum(p + r, 1e-9)))))
    model.train()
    return pd.DataFrame(rows)

_vr = np.random.default_rng(123)
SYN_VAL = [synthetic_item(_vr, 0.7) for _ in range(8 if SMOKE else 200)]

@torch.no_grad()
def eval_synthetic(items=SYN_VAL, batch_size=8):
    model.eval()
    ys, ps, ty, tp = [], [], [], []
    for i in range(0, len(items), batch_size):
        b = {k: v.to(DEVICE) for k, v in collate(items[i:i + batch_size]).items()}
        out = model(b["values"], b["delta_t"], b["time_mask"], b["channel_mask"])
        valid = (b["time_mask"][:, :, None] & b["channel_mask"][:, None, :])
        ys.append(b["labels"][valid].cpu().numpy()); ps.append(torch.sigmoid(out["logits"])[valid].float().cpu().numpy())
        tm = valid & (b["labels"] > 0) & (b["types"] > 0)
        ty.append(b["types"][tm].cpu().numpy()); tp.append(out["type_logits"][tm].argmax(-1).cpu().numpy())
    model.train()
    y, p = np.concatenate(ys), np.concatenate(ps)
    return dict(cell_auc_pr=average_precision_score(y, p), cell_auc_roc=roc_auc_score(y, p),
                type_acc=float((np.concatenate(ty) == np.concatenate(tp)).mean()) if len(ty) else float("nan"))

print(eval_synthetic())

## Eğitim

In [ ]:
import time
from torch.optim.lr_scheduler import LambdaLR

state = {"step": 0}
def difficulty():
    return min(1.0, state["step"] / max(1, STEPS * CURRICULUM))

loader = torch.utils.data.DataLoader(TrainStream(difficulty), batch_size=BATCH, collate_fn=collate,
                                     num_workers=0 if SMOKE else 4, persistent_workers=not SMOKE, prefetch_factor=None if SMOKE else 4)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05, betas=(0.9, 0.98))
sched = LambdaLR(opt, lambda s: min(1.0, (s + 1) / WARMUP) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / STEPS))))

wandb = None
if os.environ.get("WANDB_API_KEY") and not SMOKE:
    import wandb as _wb; wandb = _wb; wandb.init(project="tisan-anomali", config={**CFG, "steps": STEPS, "batch": BATCH, "lr": LR})

best, t0, run_loss = -1.0, time.time(), 0.0
os.makedirs("ckpt", exist_ok=True)
model.train()
for batch in loader:
    b = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=AMP):
        out = model(**b)
    loss = out["loss"]
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); sched.step()
    state["step"] += 1; s = state["step"]
    run_loss = 0.98 * run_loss + 0.02 * loss.item() if s > 1 else loss.item()
    if s % 50 == 0:
        print(f"adım {s:6d} | kayıp {run_loss:.4f} | zorluk {difficulty():.2f} | lr {sched.get_last_lr()[0]:.2e} | {time.time()-t0:.0f}s")
        if wandb: wandb.log({"loss": run_loss, "difficulty": difficulty(), "lr": sched.get_last_lr()[0]}, step=s)
    if s % EVAL_EVERY == 0 or s == STEPS:
        syn = eval_synthetic()
        real = eval_real(val_series, max_rows=EVAL_MAX_ROWS)
        score = float((real.auc_pr.mean() + real.auc_roc.mean()) / 2) if len(real) else syn["cell_auc_pr"]
        print(f"  ↳ sentetik {syn} | gerçek AUC-PR {real.auc_pr.mean():.3f} AUC-ROC {real.auc_roc.mean():.3f} (n={len(real)})")
        if wandb: wandb.log({**{f"syn/{k}": v for k, v in syn.items()}, "val/auc_pr": real.auc_pr.mean(), "val/auc_roc": real.auc_roc.mean()}, step=s)
        if score > best:
            best = score; model.save_pretrained("ckpt/best"); print("  ↳ en iyi model kaydedildi")
    if s >= STEPS:
        break
print("bitti; en iyi doğrulama skoru (AUC-PR+AUC-ROC)/2:", round(best, 4))

## Teşhis: seri bazında doğrulama ve satır toplulaştırma

Satır skoru = hücre skorlarının birleşimi. `max` çok sütunlu seride (HAI: 60 sütun) tek bir yanlış pozitif hücreyi satıra taşır.
Aynı checkpoint'le üç yöntem karşılaştırılır, en iyisi `config.row_agg` olarak modele yazılır; eğitim gerekmez.

In [ ]:
model = AnomaliModel.from_pretrained("ckpt/best").to(DEVICE).eval()
_cell_cache.clear()
karsilastirma = {}
for agg, k in [("max", 1), ("topk", 3), ("topk", 5), ("noisy_or", 1)]:
    r = eval_real(val_series, max_rows=EVAL_MAX_ROWS, agg=agg, topk=k, cache=True)
    karsilastirma[f"{agg}{k if agg == 'topk' else ''}"] = r.assign(yontem=f"{agg}{k if agg == 'topk' else ''}")
ozet = pd.concat(karsilastirma.values()).groupby("yontem").agg(auc_pr=("auc_pr", "mean"), auc_roc=("auc_roc", "mean"), best_f1=("best_f1", "mean")).round(3)
print(ozet)
en_iyi = ozet.assign(s=(ozet.auc_pr + ozet.auc_roc) / 2).s.idxmax()
model.config.row_agg = "noisy_or" if en_iyi == "noisy_or" else ("max" if en_iyi == "max" else "topk")
model.config.row_topk = int(en_iyi[4:]) if en_iyi.startswith("topk") else 3
print("seçilen:", model.config.row_agg, model.config.row_topk)
print()
print("seri bazında (seçilen yöntem):")
print(karsilastirma[en_iyi].drop(columns="yontem").sort_values("auc_roc").round(3).to_string(index=False))
model.save_pretrained("ckpt/best")

In [ ]:
import matplotlib.pyplot as plt
# en kötü ve en iyi doğrulama serisi: skor vs etiket
tab = karsilastirma[en_iyi].sort_values("auc_roc")
for sid in [tab.series_id.iloc[0], tab.series_id.iloc[-1]]:
    prob, y = _cell_cache[sid]
    s = aggregate_rows(prob, model.config.row_agg, model.config.row_topk)
    n = min(len(s), 100_000)
    fig, ax = plt.subplots(2, 1, figsize=(14, 4), sharex=True)
    ax[0].plot(s[:n], lw=0.5); ax[0].set_ylabel("satır skoru"); ax[0].set_title(sid)
    ax[1].fill_between(np.arange(n), 0, y[:n], color="red", alpha=0.5); ax[1].set_ylabel("etiket")
    plt.tight_layout(); plt.show()
    # hangi sütunlar en çok alarm veriyor?
    fp = (prob[:n][y[:n] == 0] > 0.7).mean(0)
    print(sid, "normal satırlarda en çok alarm veren sütunlar (yanlış pozitif oranı):", {int(i): round(float(fp[i]), 3) for i in np.argsort(fp)[-5:][::-1]})

## Kalibrasyon (temperature scaling)
Doğrulama serilerinin hücre logit'leri + sentetik doğrulama seti üzerinde NLL en aza indirilir; kısa pencereler dahil.

In [ ]:
model.eval()

@torch.no_grad()
def collect_logits(items, batch_size=8):
    L, Y = [], []
    for i in range(0, len(items), batch_size):
        b = {k: v.to(DEVICE) for k, v in collate(items[i:i + batch_size]).items()}
        out = model(b["values"], b["delta_t"], b["time_mask"], b["channel_mask"])
        valid = b["time_mask"][:, :, None] & b["channel_mask"][:, None, :]
        L.append(out["logits"][valid].float().cpu()); Y.append(b["labels"][valid].float().cpu())
    return torch.cat(L), torch.cat(Y)

# gerçek doğrulama serilerinden rastgele pencereler (etiketli, enjeksiyonsuz)
_cr = np.random.default_rng(7)
cal_items = list(SYN_VAL)
for sid, (t, X, lab) in val_series.items():
    for _ in range(2 if SMOKE else 20):
        L = int(_cr.choice([64, 256, 1024, MAX_T])); L = min(L, len(t))
        a = int(_cr.integers(0, len(t) - L + 1)); k = min(X.shape[1], MAX_CH)
        Xw = X[a:a + L, :k].astype(np.float64); ok = np.nanstd(Xw, 0) > 0
        if ok.any():
            cal_items.append(pack(t[a:a + L], gen_fill(Xw[:, ok]), (lab[a:a + L, :k][:, ok] == 1).astype(np.int8), np.full((L, int(ok.sum())), -1, np.int8)))
logits, y = collect_logits(cal_items)
import torch.nn.functional as F
grid = torch.logspace(-1.3, 1.3, 105)                               # T ∈ [0.05, 20]
nll = torch.stack([F.binary_cross_entropy_with_logits(logits / T, y) for T in grid])
best_T = float(grid[nll.argmin()])
model.config.temperature = best_T
print(f"temperature = {best_T:.3f} | NLL {F.binary_cross_entropy_with_logits(logits, y).item():.4f} → {nll.min().item():.4f}")
model.save_pretrained("ckpt/best")

## Benchmark (sadece rapor)
NAB, SMAP/MSL, SMD, SKAB. Eğitimde hiç görülmedi. Point-adjust uygulanmaz.

In [ ]:
bench_series = load_series(kat[kat.rol == "benchmark"])
bench = eval_real(bench_series, max_rows=EVAL_MAX_ROWS if SMOKE else None, agg=model.config.row_agg, topk=model.config.row_topk)
tablo = bench.groupby("source").agg(seri=("series_id", "count"), auc_pr=("auc_pr", "mean"), auc_roc=("auc_roc", "mean"), best_f1=("best_f1", "mean")).round(3)
print(tablo)
bench.to_csv("ckpt/benchmark.csv", index=False)

## Hugging Face'e yükleme (remote code)

In [ ]:
from hf_model.configuration_anomali import AnomaliConfig
from hf_model.modeling_anomali import AnomaliModel
AnomaliConfig.register_for_auto_class()
AnomaliModel.register_for_auto_class("AutoModel")

card = f"""---
license: apache-2.0
tags: [time-series, anomaly-detection, zero-shot]
---
# anomali-small

Zero-shot zaman serisi anomali modeli. Girdi: `(T, 1+k)` matris, ilk sütun zaman damgası, 1–100 değer sütunu.
Eğitim/etiket/ayar gerektirmez. {sum(p.numel() for p in model.parameters())/1e6:.1f}M parametre, iki eksenli dikkat (zaman → sütun).

```python
from transformers import AutoModel
model = AutoModel.from_pretrained("{REPO_MODEL}", trust_remote_code=True)
sonuc = model.detect(matris)          # sonuc.anomaly_rows, .row_scores, .cell_scores, .events, .to_dataframe(), .plot()
```

## Benchmark (eğitimde görülmedi, point-adjust yok)
{tablo.to_markdown()}

## Sınırlar
- Geleceği tahmin etmez; sapmayı göründüğü anda işaretler. < 20 satırda robust z-skoruna düşer.
- Türler: {", ".join(model.config.type_names[1:])}. Skorlar temperature scaling ile kalibre edilmiştir (T={model.config.temperature:.2f}); satır skoru `{model.config.row_agg}` toplulaştırması.
- Eğitim verisi: LOTSA, HAI, açık imalat/enerji SCADA setleri + çok alanlı sentetik anomaliler.
"""
if PUSH:
    model.push_to_hub(REPO_MODEL, private=True, commit_message="eğitim çıktısı")
    HfApi().upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=REPO_MODEL, repo_type="model")
    print("yüklendi:", f"https://huggingface.co/{REPO_MODEL}")
else:
    model.save_pretrained("ckpt/hub"); open("ckpt/hub/README.md", "w").write(card); print("PUSH=0: ckpt/hub altına kaydedildi")

## Kullanım örneği

In [ ]:
from transformers import AutoModel
m = AutoModel.from_pretrained(REPO_MODEL if PUSH else "ckpt/hub", trust_remote_code=True).to(DEVICE).eval()
sid, (t, X, lab) = next(iter(val_series.items()))
n = min(len(t), 6000)
sonuc = m.detect(np.column_stack([t[:n], X[:n, :8]]))
print(sid, sonuc, "| gerçek anomali satırı:", int((lab[:n] == 1).any(1).sum()))
for e in sonuc.events[:5]:
    print(e)
try:
    sonuc.plot()
except Exception as ex:
    print("çizim atlandı:", ex)